# 🧹 02. Gameplay Summary & Cleaning

### 🔍 Objective: Analyze player activity, filter out short sessions (duration cap), and trim long sessions.


### 📥 Input Data:
- `../../data/processed/merged_telemetry.csv`
<br>
### 📤 Output Artifacts:
- `../../data/processed/2_cleaned_telemetry_for_modelling.csv`- `../../data/processed/2_player_summary.csv`


In [1]:
import pandas as pd
import os

# Inputs & Outputs
INPUT_PATH = '../../data/processed/merged_telemetry.csv'
OUT_CLEANED = '../../data/processed/2_cleaned_telemetry_for_modelling.csv'
OUT_SUMMARY = '../../data/processed/2_player_summary.csv'

cap_duration = 45
min_duration = 20

print("Libraries imported and paths defined.")

Libraries imported and paths defined.


In [2]:
print("Loading processed telemetry data...")
if os.path.exists(INPUT_PATH):
    df = pd.read_csv(INPUT_PATH)
    print(f"Loaded {len(df)} rows. Unique Users: {df['userId'].nunique()}")

    # Debug timestamps
    print(f"Timestamp Dtype: {df['timestamp'].dtype}")
    if df['timestamp'].dtype == 'object':
        print("WARNING: Timestamp loaded as object. Sample:")
        print(df['timestamp'].head())
    else:
        print("Timestamp is numeric.")
else:
    raise FileNotFoundError(f"File not found at {INPUT_PATH}")

Loading processed telemetry data...
Loaded 4297 rows. Unique Users: 64
Timestamp Dtype: float64
Timestamp is numeric.


### 🧮 Calculate Player DurationCalculates the total duration played by each user based on the number of telemetry rows (assuming 30s intervals).


In [3]:
if 'userId' in df.columns:
    # Group keys: userId
    user_stats = df.groupby('userId').size().reset_index(name='total_rows')
    user_stats['duration_min'] = (user_stats['total_rows'] * 30) / 60
    
    print("\n--- Initial Duration Stats ---")
    print(user_stats['duration_min'].describe())
else:
    raise ValueError("'userId' column missing in dataset.")


--- Initial Duration Stats ---
count     64.000000
mean      33.570312
std       31.938168
min        1.000000
25%       13.250000
50%       32.250000
75%       38.625000
max      194.500000
Name: duration_min, dtype: float64


### 🔍 Filter Users (Minimum Duration)
**Constraint**: Filter out users who played for less than 20 minutes.


In [4]:
valid_users = user_stats[user_stats['duration_min'] >= min_duration]
dropped_users = user_stats[user_stats['duration_min'] < min_duration]

print(f"Filtering Rule: Keep users with >= {min_duration} minutes of play.")
print(f"Keeping: {len(valid_users)} users.")
print(f"Dropping: {len(dropped_users)} users.")

Filtering Rule: Keep users with >= 20 minutes of play.
Keeping: 45 users.
Dropping: 19 users.


### ✂️ Trim Sessions (Maximum Duration) 
**Constraint**: Cap analysis to the first 45 minutes of gameplay to ensure comparability.


In [5]:
max_rows = int((cap_duration * 60) / 30) # 90 rows

print("Applying filtering and trimming...")
# Only keep rows for valid users
df_filtered = df[df['userId'].isin(valid_users['userId'])].copy()

# Sort properly to ensure we keep the FIRST 45 mins
t_col = 'timestamp' if 'timestamp' in df_filtered.columns else 'time'

if t_col in df_filtered.columns:
     df_filtered['sort_helper'] = pd.to_numeric(df_filtered[t_col], errors='coerce')
     df_filtered.sort_values(by=['userId', 'sort_helper'], inplace=True)
     df_filtered.drop(columns=['sort_helper'], inplace=True)
# Trim > 45 mins (Keep first 90 rows per user)
df_final = df_filtered.groupby('userId').head(max_rows)

print(f"Trimming: Reduced rows from {len(df_filtered)} to {len(df_final)} (Cap at {cap_duration}m / {max_rows} rows).")

Applying filtering and trimming...
Trimming: Reduced rows from 4045 to 3240 (Cap at 45m / 90 rows).


In [6]:
# Generate stats for trimmed data
final_stats = df_final.groupby('userId').size().reset_index(name='Data points')
final_stats['Duration (minutes)'] = (final_stats['Data points'] * 30) / 60
final_stats.rename(columns={'userId': 'Player ID'}, inplace=True)
final_stats = final_stats[['Player ID', 'Duration (minutes)', 'Data points']]
final_stats.sort_values('Duration (minutes)', ascending=False, inplace=True)

print("\n--- Final Cleaned Summary (Top 5) ---")
print(final_stats.head(5).to_string(index=False))

# Export
df_final.to_csv(OUT_CLEANED, index=False)
final_stats.to_csv(OUT_SUMMARY, index=False)

print(f"\nSaved cleaned dataset to: {OUT_CLEANED}")
print(f"Saved summary to: {OUT_SUMMARY}")
print("Gameplay summary processing completed.")


--- Final Cleaned Summary (Top 5) ---
               Player ID  Duration (minutes)  Data points
6974892348d53c4152cf1421                45.0           90
6974acd8315a39e91bc1e52f                45.0           90
6974f3cc5d8e98b89f0a5d07                45.0           90
697502455d8e98b89f0a9f38                45.0           90
69757efd2890de71026a214a                45.0           90

Saved cleaned dataset to: ../../data/processed/2_cleaned_telemetry_for_modelling.csv
Saved summary to: ../../data/processed/2_player_summary.csv
Gameplay summary processing completed.
